# Práctica de SQL: análisis de películas

**Objetivo:** practicar consultas SQL sobre películas utilizando SQLite y pandas, desde filtros básicos hasta agregaciones y clasificación con `CASE`.

## Cómo ejecutar

- Coloca este notebook en `notebooks/` y conserva el archivo `data/raw/movies.csv` del proyecto.
- Ejecuta el notebook desde la carpeta `notebooks/`, con Python y pandas instalados. `sqlite3` forma parte de Python.
- Reinicia el kernel y ejecuta todas las celdas en orden. La base SQLite se crea en memoria y no modifica el CSV.

Los ejercicios por género usan el período **2000–2010, ambos años incluidos**. Las consultas de nulos y países utilizan toda la tabla. Esta entrega cubre consultas sobre una tabla; los JOIN quedan para la siguiente práctica.


## 1. Carga de datos

Lectura del CSV e inspección de las primeras filas.

In [2]:
import pandas as pd
import sqlite3

movies = pd.read_csv('../data/raw/movies.csv')

movies.head()

,name,rating,genre,year,released,score,votes,director,writer,star,country,budget,gross,company,runtime
0,The Shining,R,Drama,1980,"June 13, 1980 (United States)",8.4,927000.0,Stanley Kubrick,Stephen King,Jack Nicholson,United Kingdom,19000000.0,46998772.0,Warner Bros.,146.0
1,The Blue Lagoon,R,Adventure,1980,"July 2, 1980 (United States)",5.8,65000.0,Randal Kleiser,Henry De Vere Stacpoole,Brooke Shields,United States,4500000.0,58853106.0,Columbia Pictures,104.0
2,Star Wars: Episode V - The Empire Strikes Back,PG,Action,1980,"June 20, 1980 (United States)",8.7,1200000.0,Irvin Kershner,Leigh Brackett,Mark Hamill,United States,18000000.0,538375067.0,Lucasfilm,124.0
3,Airplane!,PG,Comedy,1980,"July 2, 1980 (United States)",7.7,221000.0,Jim Abrahams,Jim Abrahams,Robert Hays,United States,3500000.0,83453539.0,Paramount Pictures,88.0
4,Caddyshack,R,Comedy,1980,"July 25, 1980 (United States)",7.3,108000.0,Harold Ramis,Brian Doyle-Murray,Chevy Chase,United States,6000000.0,39846344.0,Orion Pictures,98.0


## 2. Base de datos en memoria

Cada fila del CSV se carga en la tabla `movies`, sin añadir el índice de pandas.

In [3]:
conn = sqlite3.connect(':memory:')

movies.to_sql('movies', conn, index = False, if_exists= 'replace')


7668

## 3. Selección y filtros básicos

Selección de columnas y películas según su puntuación. `LIMIT` sin `ORDER BY` muestra una muestra sin garantizar un orden.

In [4]:
query = """
SELECT * 
FROM movies
LIMIT 5;
"""

In [5]:
pd.read_sql_query(query, conn)

,name,rating,genre,year,released,score,votes,director,writer,star,country,budget,gross,company,runtime
0,The Shining,R,Drama,1980,"June 13, 1980 (United States)",8.4,927000.0,Stanley Kubrick,Stephen King,Jack Nicholson,United Kingdom,19000000.0,46998772.0,Warner Bros.,146.0
1,The Blue Lagoon,R,Adventure,1980,"July 2, 1980 (United States)",5.8,65000.0,Randal Kleiser,Henry De Vere Stacpoole,Brooke Shields,United States,4500000.0,58853106.0,Columbia Pictures,104.0
2,Star Wars: Episode V - The Empire Strikes Back,PG,Action,1980,"June 20, 1980 (United States)",8.7,1200000.0,Irvin Kershner,Leigh Brackett,Mark Hamill,United States,18000000.0,538375067.0,Lucasfilm,124.0
3,Airplane!,PG,Comedy,1980,"July 2, 1980 (United States)",7.7,221000.0,Jim Abrahams,Jim Abrahams,Robert Hays,United States,3500000.0,83453539.0,Paramount Pictures,88.0
4,Caddyshack,R,Comedy,1980,"July 25, 1980 (United States)",7.3,108000.0,Harold Ramis,Brian Doyle-Murray,Chevy Chase,United States,6000000.0,39846344.0,Orion Pictures,98.0


In [6]:
query = """
SELECT name,
     genre,
     score 
FROM movies
LIMIT 5;
"""

pd.read_sql_query( query, conn)

,name,genre,score
0,The Shining,Drama,8.4
1,The Blue Lagoon,Adventure,5.8
2,Star Wars: Episode V - The Empire Strikes Back,Action,8.7
3,Airplane!,Comedy,7.7
4,Caddyshack,Comedy,7.3


In [7]:
query = """
SELECT name,
     genre,
     score 
FROM movies
WHERE score>= 8
LIMIT 5;
"""

pd.read_sql_query(query, conn)

,name,genre,score
0,The Shining,Drama,8.4
1,Star Wars: Episode V - The Empire Strikes Back,Action,8.7
2,Raging Bull,Biography,8.2
3,Lion of the Desert,Biography,8.3
4,Moscow Does Not Believe in Tears,Comedy,8.1


## 4. Función auxiliar y filtros combinados

La función `sql()` devuelve el resultado como DataFrame. Se practican `AND`, `OR`, `IN`, `BETWEEN` y orden descendente. Los empates no tienen un orden definido si no se añade un segundo criterio.

In [8]:
def sql(query):
    return pd.read_sql_query(query, conn)

In [9]:
sql("""
SELECT name,
    genre,
     score
FROM movies
WHERE genre = 'Comedy'
LIMIT 5;
""")

,name,genre,score
0,Airplane!,Comedy,7.7
1,Caddyshack,Comedy,7.3
2,9 to 5,Comedy,6.9
3,Stir Crazy,Comedy,6.8
4,Little Darlings,Comedy,6.5


In [10]:
sql("""
SELECT name,
     genre,
     score
FROM movies
WHERE genre ='Comedy' AND score >=8
LIMIT 5;
""")

,name,genre,score
0,Moscow Does Not Believe in Tears,Comedy,8.1
1,Tampopo,Comedy,8.0
2,Jean de Florette,Comedy,8.0
3,Time of the Gypsies,Comedy,8.2
4,Dead Poets Society,Comedy,8.1


In [11]:
sql("""
SELECT name,
     genre,
     score
FROM movies
WHERE genre ='Comedy' AND score >= 8
ORDER BY score DESC
LIMIT 5;
""")

,name,genre,score
0,Life Is Beautiful,Comedy,8.6
1,Parasite,Comedy,8.6
2,3 Idiots,Comedy,8.4
3,Snatch,Comedy,8.3
4,Amélie,Comedy,8.3


In [12]:
sql("""
SELECT name,
     genre,
     score
FROM movies
WHERE genre = 'Comedy' OR score >= 8
ORDER BY score DESC
LIMIT 10;
""")

,name,genre,score
0,The Shawshank Redemption,Drama,9.3
1,The Dark Knight,Action,9.0
2,Schindler's List,Biography,8.9
3,Pulp Fiction,Crime,8.9
4,The Lord of the Rings: The Return of the King,Action,8.9
5,Forrest Gump,Drama,8.8
6,Fight Club,Drama,8.8
7,The Lord of the Rings: The Fellowship of the Ring,Action,8.8
8,Inception,Action,8.8
9,Star Wars: Episode V - The Empire Strikes Back,Action,8.7


In [13]:
sql("""
SELECT name,
     genre,
     score
FROM movies
WHERE genre ='Comedy' OR genre = 'Action'
ORDER BY score DESC
LIMIT 10;
""")

,name,genre,score
0,The Dark Knight,Action,9.0
1,The Lord of the Rings: The Return of the King,Action,8.9
2,The Lord of the Rings: The Fellowship of the Ring,Action,8.8
3,Inception,Action,8.8
4,Star Wars: Episode V - The Empire Strikes Back,Action,8.7
5,The Matrix,Action,8.7
6,The Lord of the Rings: The Two Towers,Action,8.7
7,Life Is Beautiful,Comedy,8.6
8,Parasite,Comedy,8.6
9,Terminator 2: Judgment Day,Action,8.5


In [14]:
sql("""
SELECT name,
     genre,
     score
FROM movies
WHERE genre IN('Comedy', 'Action', 'Drama')
ORDER BY score DESC
LIMIT 10;
""")

,name,genre,score
0,The Shawshank Redemption,Drama,9.3
1,The Dark Knight,Action,9.0
2,The Lord of the Rings: The Return of the King,Action,8.9
3,Forrest Gump,Drama,8.8
4,Fight Club,Drama,8.8
5,The Lord of the Rings: The Fellowship of the Ring,Action,8.8
6,Inception,Action,8.8
7,Star Wars: Episode V - The Empire Strikes Back,Action,8.7
8,The Matrix,Action,8.7
9,The Lord of the Rings: The Two Towers,Action,8.7


In [15]:
sql("""
SELECT name,
     genre,
     score,
     year
FROM movies
WHERE year BETWEEN 2000 AND 2010
ORDER BY score DESC
LIMIT 10;
""" )

,name,genre,score,year
0,The Dark Knight,Action,9.0,2008
1,The Lord of the Rings: The Return of the King,Action,8.9,2003
2,The Lord of the Rings: The Fellowship of the Ring,Action,8.8,2001
3,Inception,Action,8.8,2010
4,The Lord of the Rings: The Two Towers,Action,8.7,2002
5,Spirited Away,Animation,8.6,2001
6,City of God,Crime,8.6,2002
7,Gladiator,Action,8.5,2000
8,The Pianist,Biography,8.5,2002
9,The Best of Youth,Drama,8.5,2003


In [16]:
sql("""
SELECT name,
     genre,
     score,
     year
FROM movies
WHERE year BETWEEN 2000 AND 2010 AND genre IN ('Action', 'Drama')
ORDER BY score DESC
LIMIT 5;
"""
)

,name,genre,score,year
0,The Dark Knight,Action,9.0,2008
1,The Lord of the Rings: The Return of the King,Action,8.9,2003
2,The Lord of the Rings: The Fellowship of the Ring,Action,8.8,2001
3,Inception,Action,8.8,2010
4,The Lord of the Rings: The Two Towers,Action,8.7,2002


## 5. Agregaciones por género

`COUNT(*)` cuenta películas y `AVG(score)` calcula la media de puntuaciones disponibles. `GROUP BY` forma los grupos.

In [17]:
sql("""
SELECT COUNT(*) AS total_movies
FROM movies
WHERE year BETWEEN 2000 AND 2010 AND genre IN ('Action' , 'Drama');
""")

,total_movies
0,916


In [18]:
sql ("""
SELECT genre,
     COUNT(*) AS total_movies
FROM movies
WHERE year BETWEEN 2000 AND 2010 AND genre IN ('Action' , 'Drama')
GROUP BY genre;
""")

,genre,total_movies
0,Action,498
1,Drama,418


In [19]:
sql ("""
SELECT genre,
     COUNT(*) AS total_movies,
     AVG(score) AS average_score
FROM movies 
WHERE year BETWEEN 2000 AND 2010 AND genre IN ('Action' , 'Drama')
GROUP BY genre;
""")

,genre,total_movies,average_score
0,Action,498,6.270683
1,Drama,418,6.764115


In [20]:
sql ("""
SELECT genre,
     COUNT(*) AS total_movies,
     AVG(score) AS average_score
FROM movies
WHERE year BETWEEN 2000 AND 2010
GROUP BY genre
ORDER BY average_score DESC
LIMIT 3;
""")

,genre,total_movies,average_score
0,Mystery,5,7.16000
1,Biography,122,7.07459
2,Thriller,2,6.95000


## 6. Filtrado de grupos con HAVING

`WHERE` filtra películas antes de agrupar; `HAVING` filtra grupos después. SQLite permite los alias del SELECT en HAVING; este comportamiento puede variar entre motores SQL.

In [21]:
sql ("""
SELECT genre,
     COUNT(*) AS total_movies,
     AVG(score) AS average_score
FROM movies
WHERE year BETWEEN 2000 AND 2010    
GROUP BY genre
HAVING COUNT(*)>10
ORDER BY average_score DESC
LIMIT 3;
""")

,genre,total_movies,average_score
0,Biography,122,7.074590
1,Drama,418,6.764115
2,Crime,182,6.737912


In [22]:
sql ("""
SELECT genre,
     COUNT(*) AS total_movies
FROM movies
WHERE year BETWEEN 2000 AND 2010
GROUP BY genre
ORDER BY total_movies DESC;
""")

,genre,total_movies
0,Comedy,631
1,Action,498
2,Drama,418
3,Crime,182
4,Biography,122
5,Animation,120
6,Adventure,118
7,Horror,90
8,Fantasy,8
9,Mystery,5


In [23]:
sql ("""
SELECT genre,
     COUNT(*) AS total_movies
FROM movies
WHERE year BETWEEN 2000 AND 2010
GROUP BY genre
HAVING total_movies>= 100
ORDER BY total_movies DESC;
""")

,genre,total_movies
0,Comedy,631
1,Action,498
2,Drama,418
3,Crime,182
4,Biography,122
5,Animation,120
6,Adventure,118


In [24]:
sql ("""
SELECT genre,
     COUNT(*) AS total_movies,
     AVG(score) AS average_score
FROM movies
WHERE year BETWEEN 2000 AND 2010
GROUP BY genre
HAVING total_movies>= 100 AND average_score >=6.5
ORDER BY average_score DESC;
""")

,genre,total_movies,average_score
0,Biography,122,7.074590
1,Drama,418,6.764115
2,Crime,182,6.737912
3,Animation,120,6.719167


In [25]:
sql("""
SELECT genre,
     COUNT(*) AS total_movies,
     AVG(score) AS average_score
FROM movies
WHERE year BETWEEN 2000 AND 2010
GROUP BY genre
HAVING average_score>= 7 AND total_movies>= 50
ORDER BY average_score DESC;
""")

,genre,total_movies,average_score
0,Biography,122,7.07459


## 7. Recaudación por género

Se comparan suma, promedio, mínimo, máximo y rango. El rango es `MAX(gross) - MIN(gross)` y solo describe la distancia entre los extremos. Los importes se expresan en las unidades del archivo; las columnas terminadas en `_millions` los dividen entre un millón.

In [26]:
sql("""
SELECT genre,
     SUM(gross) AS total_gross     
FROM movies
WHERE year BETWEEN 2000 AND 2010
GROUP BY genre
ORDER BY total_gross DESC
LIMIT 3;
""")

,genre,total_gross
0,Action,7.782396e+10
1,Comedy,4.029460e+10
2,Animation,2.784710e+10


In [27]:
sql("""
SELECT genre,
     COUNT(*) AS total_movies,
     AVG(gross) AS average_gross,
     SUM(gross) AS total_gross
FROM movies
WHERE year BETWEEN 2000 AND 2010
GROUP BY genre
ORDER BY total_gross DESC
LIMIT 3;
""" )

,genre,total_movies,average_gross,total_gross
0,Action,498,1.565874e+08,7.782396e+10
1,Comedy,631,6.395969e+07,4.029460e+10
2,Animation,120,2.320591e+08,2.784710e+10


In [28]:
sql("""
SELECT genre,
     ROUND(MIN(gross) / 1000000.0, 2) AS min_gross_millions,
     ROUND(MAX(gross)/ 1000000.0, 2) AS max_gross_millions,
     ROUND(AVG(gross) / 1000000.0,2) AS average_gross_millions,
     ROUND((MAX(gross) - MIN(gross))/ 1000000.0,2) AS gross_range_millions
FROM movies
WHERE year BETWEEN 2000 AND 2010
GROUP BY genre
ORDER BY gross_range_millions DESC
LIMIT 5;
""")

,genre,min_gross_millions,max_gross_millions,average_gross_millions,gross_range_millions
0,Action,0.01,2847.25,156.59,2847.23
1,Animation,0.04,1066.97,232.06,1066.93
2,Adventure,0.12,1025.47,154.22,1025.35
3,Mystery,23.73,760.01,241.77,736.28
4,Drama,0.00,612.05,47.09,612.05


## 8. Datos faltantes

`COUNT(*)` incluye todas las filas; `COUNT(gross)` cuenta únicamente recaudaciones no nulas. Su diferencia indica cuántas películas no tienen ese dato. `AVG`, `SUM`, `MIN` y `MAX` también omiten los nulos.

In [29]:
sql("""
SELECT COUNT(*) AS total_movies,
     COUNT(gross) AS movies_with_gross,
     (COUNT(*) - COUNT(gross)) AS movies_without_gross
FROM movies;
""")

,total_movies,movies_with_gross,movies_without_gross
0,7668,7479,189


## 9. Países únicos

`DISTINCT` elimina repeticiones del resultado. La lista puede incluir un NULL, pero `COUNT(DISTINCT country)` no lo cuenta.

In [30]:
sql("""
SELECT DISTINCT country
FROM movies
ORDER BY country;
""")

,country
0,None
1,Argentina
2,Aruba
3,Australia
4,Austria
5,Belgium
6,Brazil
7,Canada
8,Chile
9,China


In [32]:
sql("""
SELECT COUNT(DISTINCT country) AS total_countries
FROM movies;
""")

,total_countries
0,59


## 10. Clasificación con CASE

Primero se construyen dos categorías y después se distingue entre datos faltantes, puntuación alta (7 o más), media (5 a menos de 7) y baja (menos de 5). CASE devuelve la primera condición verdadera.

In [35]:
sql("""
SELECT name,
     score,
     CASE
        WHEN score >= 7 THEN 'Alto'
        ELSE 'Otra'
        END AS score_category
FROM movies
LIMIT 10;
""")

,name,score,score_category
0,The Shining,8.4,Alto
1,The Blue Lagoon,5.8,Otra
2,Star Wars: Episode V - The Empire Strikes Back,8.7,Alto
3,Airplane!,7.7,Alto
4,Caddyshack,7.3,Alto
5,Friday the 13th,6.4,Otra
6,The Blues Brothers,7.9,Alto
7,Raging Bull,8.2,Alto
8,Superman II,6.8,Otra
9,The Long Riders,7.0,Alto


In [ ]:
sql("""
SELECT name,
     score,
     CASE
        WHEN score IS NULL THEN 'Sin dato'
        WHEN score >=7 THEN 'Alto'
        WHEN score >=5 THEN 'Medio'
        ELSE 'Bajo'
        END AS score_category
FROM movies
LIMIT 10;
""")

,name,score,score_category
0,The Shining,8.4,Alto
1,The Blue Lagoon,5.8,Medio
2,Star Wars: Episode V - The Empire Strikes Back,8.7,Alto
3,Airplane!,7.7,Alto
4,Caddyshack,7.3,Alto
5,Friday the 13th,6.4,Medio
6,The Blues Brothers,7.9,Alto
7,Raging Bull,8.2,Alto
8,Superman II,6.8,Medio
9,The Long Riders,7.0,Alto


## Conclusiones

- Entre 2000 y 2010, Action y Drama reúnen 916 películas: 498 y 418, respectivamente.
- Action lidera la recaudación total por género en ese período. Animation tiene un promedio por película mayor que Action (aproximadamente 232,06 frente a 156,59 millones), sin que eso implique que cada película animada recaude más.
- Biography es el único género que cumple simultáneamente un promedio de puntuación de al menos 7 y un mínimo de 50 películas en el período analizado.
- En toda la tabla hay 7668 películas: 7479 tienen recaudación registrada y 189 no. Se registran 59 países distintos, excluyendo los nulos.
- Los promedios de recaudación se calculan sobre valores disponibles: el denominador puede ser menor que COUNT(*). Un mínimo mostrado como 0,00 millones puede deberse al redondeo.

### Alcance

Los resultados describen este archivo y los filtros utilizados; no permiten garantizar la recaudación de películas futuras. No se han ajustado los importes por inflación. Un rango grande no describe por sí solo cómo se distribuyen los valores intermedios.
